# Lezione 14 — Web Scraping con BeautifulSoup

## Obiettivi della lezione

Al termine di questa lezione sarai in grado di:

- Spiegare cos'è il web scraping e perché si usa quando non esiste un'API.
- Convertire un blocco di HTML in un oggetto navigabile con `BeautifulSoup` e visualizzarlo con `prettify()`.
- Capire cos'è un `Tag`, accedere ai suoi **attributi** e al suo contenuto testuale (`NavigableString`).
- Navigare l'albero HTML con `parent`, `next_sibling` e scorciatoie come `soup.h3`.
- Usare `find()` e `find_all()` con diversi filtri (nome, attributi, `class_`, testo) per localizzare elementi.
- Estrarre una tabella HTML "a mano", scorrendo righe (`find_all('tr')`) e celle (`find_all('td')`).
- Estrarre la stessa tabella con la scorciatoia di pandas `pd.read_html()`.

> Requisito: conosci già Python (funzioni, cicli, dizionari), pandas di base e come richiedere dati con `requests` a un'API. Non serve connessione a internet per questa lezione — tutto l'HTML che usiamo è definito come testo direttamente nel notebook.

## Perché ci interessa?

Quando un'API esiste, è sempre l'opzione migliore per ottenere dati: ti dà informazioni pulite e strutturate. Il problema è che la maggior parte dei dati pubblici su internet **non ha un'API dietro** — sono incorporati nell'HTML di una pagina, pensati perché li legga una persona in un browser, non una macchina.

Il **web scraping** è la tecnica di prendere quell'HTML ed estrarne i dati che ti interessano, automatizzando quello che faresti a mano con copia e incolla. `BeautifulSoup` è la libreria Python che converte l'HTML (un blocco di testo) in una struttura navigabile di oggetti, così puoi percorrerla e filtrarla con codice invece di leggerla tu stesso.

**Nota sull'installazione:** `BeautifulSoup` vive nel pacchetto `bs4`, che potrebbe non essere preinstallato in un Jupyter locale (a differenza di Google Colab, dove è già presente). Se eseguendo `import bs4` ottieni un `ModuleNotFoundError`, installalo con:

```
pip install beautifulsoup4
```

Più sotto c'è una cella opzionale con `!pip install beautifulsoup4` nel caso tu preferisca eseguirlo direttamente dal notebook.

**Cella opzionale** — eseguila solo se `import bs4` fallisce più sotto. Non serve eseguirla se `bs4` è già installato nel tuo ambiente.

In [ ]:
# Cella OPZIONALE: decommenta la riga seguente solo se non hai bs4 installato
# !pip install beautifulsoup4

## Il nostro HTML di esempio: una tabella di prodotti

Invece di scaricare una pagina reale con `requests` (che potrebbe non essere disponibile senza connessione, o cambiare nel tempo), lavoreremo su un HTML di esempio definito direttamente come **stringa Python**. Così il notebook funziona allo stesso modo con o senza internet.

Definiamo una tabella semplice con 4 prodotti fittizi: nome, categoria, prezzo e stock.

In [ ]:
html_productos = """
<!DOCTYPE html>
<html>
<head><title>Catalogo dei prodotti</title></head>
<body>
<h3 id="titulo_tabla">Catalogo del negozio</h3>
<p>Questi sono i nostri <b id="destacado">prodotti in evidenza</b> di questa settimana.</p>
<table id="tabla_productos" class="catalogo">
  <tr>
    <th>Nome</th>
    <th>Categoria</th>
    <th>Prezzo</th>
    <th>Stock</th>
  </tr>
  <tr>
    <td>Mouse wireless</td>
    <td>Accessori</td>
    <td>$15.99</td>
    <td>120</td>
  </tr>
  <tr>
    <td>Tastiera meccanica</td>
    <td>Accessori</td>
    <td>$49.50</td>
    <td>45</td>
  </tr>
  <tr>
    <td>Monitor 24 pollici</td>
    <td><a href="https://ejemplo.com/categoria/pantallas">Schermi</a></td>
    <td>$189.00</td>
    <td>10</td>
  </tr>
  <tr>
    <td>Cuffie Bluetooth</td>
    <td>Audio</td>
    <td>$34.75</td>
    <td>0</td>
  </tr>
</table>
</body>
</html>
"""

print(html_productos[:120], "...")

## Creare l'oggetto `BeautifulSoup`

Per fare il parsing dell'HTML lo passiamo al costruttore `BeautifulSoup(html, parser)`. Il secondo argomento indica quale "motore" di parsing usare; `'html.parser'` è incluso nella libreria standard di Python, quindi non richiede installazione aggiuntiva (a differenza di `'html5lib'` o `'lxml'`, alternative più tolleranti verso HTML mal formato ma che richiedono un'installazione a parte).

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_productos, 'html.parser')
print(type(soup))

Con `prettify()` puoi stampare l'HTML reindentato secondo la sua gerarchia di tag — molto utile per capire visivamente come è annidata una pagina prima di scrivere il codice che la percorre.

In [ ]:
print(soup.prettify())

## Tag: accedere a un elemento dell'HTML

Quando accedi a `soup.<nome_del_tag>`, BeautifulSoup ti restituisce un oggetto `Tag` — la rappresentazione di quell'elemento. Se ci sono più tag con lo stesso nome, restituisce **il primo** che trova nel documento.

In [ ]:
tag_titulo = soup.title
print("tag titolo:", tag_titulo)
print("tipo:", type(tag_titulo))

tag_h3 = soup.h3   # il primo <h3> di tutto il documento
print("tag h3:", tag_h3)

## Navigazione: children, parent e siblings

Da qualsiasi `Tag` ti puoi muovere nelle tre direzioni dell'albero HTML:

- **Figlio** (*child*): accedendo con `.nome_del_tag` scendi di un livello.
- **Genitore** (*parent*): con `.parent` sali di un livello.
- **Fratello** (*sibling*): con `.next_sibling` ti sposti all'elemento successivo allo stesso livello.

Un dettaglio importante con `.next_sibling`: quando l'HTML originale ha ritorni a capo o spazi tra un tag e il successivo (come nel nostro `html_productos`, scritto su più righe per essere più leggibile), anche quello spazio bianco conta come un "fratello" — un `NavigableString` vuoto, non il tag che cercavi. Per saltare direttamente al **tag successivo** (ignorando gli spazi bianchi isolati), usa invece `.find_next_sibling()`.

In [ ]:
tag_p = soup.p            # il primo <p> del documento
tag_child = tag_p.b       # scendiamo di un livello: il <b> che sta dentro il <p>
print("figlio (b):", tag_child)

tag_parent = tag_child.parent   # saliamo: torniamo al <p> che lo contiene
print("genitore:", tag_parent)

# next_sibling, così com'è, può restituire un a capo invece del tag atteso
print("next_sibling 'grezzo':", repr(soup.h3.next_sibling))

# find_next_sibling() salta gli spazi bianchi e restituisce direttamente il tag successivo
siguiente = soup.h3.find_next_sibling()   # il <p> che segue l'<h3>
print("fratello successivo di h3:", siguiente)

## Attributi e contenuto testuale

Un tag con attributi si comporta come un dizionario Python: puoi accedere al valore di un attributo con le parentesi quadre, con `.get()`, oppure consultarli tutti insieme con `.attrs`.

Il testo che sta dentro un tag (non il tag in sé, ma il contenuto) è un oggetto `NavigableString` — si comporta come una stringa normale, ma appartiene alla struttura di BeautifulSoup.

In [ ]:
tag_destacado = soup.b   # il <b id="destacado"> che abbiamo visto prima

print("attributo id con parentesi quadre:", tag_destacado['id'])
print("attributo id con .get():", tag_destacado.get('id'))
print("tutti gli attributi:", tag_destacado.attrs)

tag_string = tag_destacado.string
print("contenuto:", tag_string)
print("tipo:", type(tag_string))
print("come stringa normale di Python:", str(tag_string))

## `find()` e `find_all()`: filtrare l'albero

`find_all()` percorre tutto il documento e restituisce una **lista** con tutti i tag che corrispondono al filtro. `find()` restituisce direttamente il **primo** risultato, senza racchiuderlo in una lista.

Il filtro più semplice è il nome del tag, ma si può anche filtrare per attributi, per una lista di nomi, o per il testo esatto contenuto in un elemento. Poiché `class` è una parola riservata in Python, per filtrare sull'attributo HTML `class` si usa `class_` (con trattino basso finale).

In [ ]:
# find(): il primo <td> del documento, senza lista
primer_td = soup.find('td')
print("find('td'):", primer_td)

# find_all(): TUTTI i <td> del documento
todos_los_td = soup.find_all('td')
print("quantità di <td> trovati:", len(todos_los_td))

# Filtrare per una lista di nomi di tag
tags_tr_o_a = soup.find_all(name=["tr", "a"])
print("quantità di <tr> o <a>:", len(tags_tr_o_a))

# Filtrare per attributo: il tag il cui id è "tabla_productos"
tabla_por_id = soup.find(id="tabla_productos")
print("tabella trovata per id:", tabla_por_id.name, tabla_por_id.get('id'))

# Filtrare per class_ (con trattino basso, perché class è parola riservata in Python)
tabla_por_clase = soup.find("table", class_="catalogo")
print("tabella trovata per class_:", tabla_por_clase.get('class'))

# Filtrare per href: tutti gli <a> che hanno href, indipendentemente dal valore
links_con_href = soup.find_all('a', href=True)
print("link con href:", links_con_href)

# Filtrare per testo esatto contenuto in un elemento
celdas_accesorios = soup.find_all(string="Accessori")
print("celle con il testo 'Accessori':", celdas_accesorios)

## Estrarre la tabella "a mano": `find_all('tr')` + `find_all('td')`

Lo schema per percorrere qualsiasi tabella HTML è sempre lo stesso: prima trovi la tabella con `find('table')`, poi percorri le sue righe con `find_all('tr')`, e dentro ogni riga percorri le sue celle con `find_all('td')`.

Prima di scrivere questo tipo di codice conviene guardare la tabella (qui, il `prettify()` di prima) e annotare quante colonne ha e in che ordine — così sai in anticipo quale indice di `cols[...]` corrisponde a ciascun dato, invece di indovinare. Nella nostra tabella: colonna 0 = nome, colonna 1 = categoria, colonna 2 = prezzo, colonna 3 = stock.

In [ ]:
tabla = soup.find('table')   # l'unica tabella del documento

filas = tabla.find_all('tr')
print("quantità di righe (include l'intestazione):", len(filas))

productos = []
for fila in filas[1:]:   # saltiamo la prima riga, che è l'intestazione
    cols = fila.find_all('td')
    nombre = cols[0].get_text()     # .get_text() prende il testo anche se ci sono tag dentro (come <a>)
    categoria = cols[1].get_text()
    precio = cols[2].string
    stock = cols[3].string
    productos.append((nombre, categoria, precio, stock))

for p in productos:
    print("{} | {} | {} | stock: {}".format(*p))

Nota che usiamo `.get_text()` per la colonna categoria invece di `.string`: la cella di "Monitor 24 pollici" ha un `<a>` dentro (`<td><a href="...">Schermi</a></td>`), e `.string` funziona solo quando il tag ha un unico figlio di testo diretto. `.get_text()` è più robusto perché unisce tutto il testo, indipendentemente da quanti tag ci siano annidati dentro.

## La scorciatoia di pandas: `pd.read_html()`

Quando l'unica cosa che ti serve è una tabella completa, non serve percorrerla cella per cella con BeautifulSoup — pandas offre `pd.read_html()`, che fa quel lavoro al posto tuo e restituisce una **lista di DataFrame**, uno per ogni tabella che trova.

`pd.read_html()` accetta una URL o un blocco di HTML. Da pandas 2.1, il modo consigliato per passargli una stringa HTML (invece di una URL o un file) è avvolgerla con `io.StringIO(...)` — passare la stringa diretta funziona ma genera un avviso (`FutureWarning`) nelle versioni recenti di pandas.

In [ ]:
import io
import pandas as pd

tablas = pd.read_html(io.StringIO(html_productos))
print("quantità di tabelle trovate:", len(tablas))

df_productos = tablas[0]   # c'è una sola tabella nella pagina, quindi è la prima della lista
df_productos

Confronta il risultato: in una sola riga (`pd.read_html`) hai un DataFrame pronto da analizzare con pandas, mentre a mano con BeautifulSoup hai avuto controllo totale su quale testo prendere da ogni cella (per esempio, decidere di usare `.get_text()` invece di `.string` per la cella con il link).

La regola pratica: usa `pd.read_html()` come primo tentativo quando ti serve una tabella completa — ti risparmia codice. Se la pagina non collabora (il contenuto non è costruito come `<table>` vera, o arriva con troppa spazzatura da pulire), ricorri a BeautifulSoup per avere controllo totale con `find_all('tr')` / `find_all('td')`.

## Riepilogo della lezione

- Il web scraping estrae dati direttamente dall'HTML di una pagina quando non esiste un'API che li fornisca in modo strutturato.
- `BeautifulSoup(html, 'html.parser')` converte una stringa HTML in un oggetto navigabile; `prettify()` la mostra reindentata secondo la sua gerarchia.
- Ogni elemento dell'HTML è un oggetto `Tag`; `soup.<tag>` restituisce il primo che compare nel documento.
- Si naviga l'albero con `.parent` (sale), `.next_sibling` (a fianco) e scorciatoie come `soup.h3` (scende).
- Gli attributi di un tag si accedono come un dizionario (`tag['id']`, `tag.get('id')`, `tag.attrs`); il testo interno è un `NavigableString`, simile a una stringa comune.
- `find_all()` restituisce **tutti** gli elementi che soddisfano un filtro (per nome, attributi, `class_`, `href`, testo); `find()` restituisce solo il **primo**, senza lista.
- Una tabella si percorre sempre allo stesso modo: `find('table')` per individuarla, `find_all('tr')` per le righe, e `find_all('td')` dentro ogni riga per le celle. `.get_text()` è più robusto di `.string` quando una cella ha tag annidati dentro.
- `pd.read_html(io.StringIO(html))` è la scorciatoia di pandas: converte direttamente le tabelle di un HTML in una lista di DataFrame, in una sola riga.

**Prossima lezione:** continuerai a combinare questi strumenti con progetti che integrano API, pandas e scraping per costruire dataset tuoi.